# kaggle-vllm Milestone 2 — Qwen TP1/TP2 concurrency crossover

This output-free source notebook runs the required online streaming matrix on a fresh Kaggle **T4 ×2** session. It tests whether a crossover occurs; it does not assume TP2 wins. Real GPU acceptance remains pending until the downloaded evidence bundle is reviewed.

Required settings: Accelerator **GPU T4 ×2**, Internet **On** (unless the pinned model is attached as a Kaggle Input), and an optional `HF_TOKEN` Kaggle Secret. The token is never printed or written to evidence.

In [ ]:
import hashlib, json, os, platform, shutil, subprocess, sys, tarfile, urllib.request
from pathlib import Path

REVIEWED_SOURCE_COMMIT = "4f8dcc1c032d65d54b1cce3ca213535d68fd5099"
MODEL_REPOSITORY = "Qwen/Qwen2.5-3B-Instruct"
MODEL_REVISION = "aa8e72537993ba99e69dfaafa59ed015b17504d1"
# Set this to a read-only /kaggle/input/... Transformers snapshot to run offline.
ATTACHED_MODEL_PATH = None
WORK_ROOT = Path("/kaggle/working/kaggle-vllm-milestone-2-work")
EVIDENCE_DIR = Path("/kaggle/working/kaggle-vllm-milestone-2")
EVIDENCE_ZIP = Path("/kaggle/working/kaggle-vllm-milestone-2.zip")
SOURCE_URL = f"https://github.com/kaggle-vllm/kaggle-vllm/archive/{REVIEWED_SOURCE_COMMIT}.tar.gz"

print("Reviewed source commit:", REVIEWED_SOURCE_COMMIT)
print("Pinned model:", MODEL_REPOSITORY, MODEL_REVISION)
print("Python:", sys.version)
print("Platform:", platform.platform())
assert Path("/kaggle/working").is_dir(), "Run this notebook on Kaggle"
assert not EVIDENCE_DIR.exists(), f"Refusing existing evidence: {EVIDENCE_DIR}"
assert not EVIDENCE_ZIP.exists(), f"Refusing existing ZIP: {EVIDENCE_ZIP}"

In [ ]:
import torch

print("Torch:", torch.__version__, torch.__file__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
for command in (["nvidia-smi"], ["nvidia-smi", "topo", "-m"]):
    print("$", " ".join(command), flush=True)
    subprocess.run(command, check=True)
assert torch.__version__ == "2.10.0+cu128"
assert torch.version.cuda == "12.8"
assert torch.cuda.device_count() == 2
assert all("Tesla T4" in torch.cuda.get_device_name(i) for i in range(2))
assert all(torch.cuda.get_device_capability(i) == (7, 5) for i in range(2))

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def safe_extract(archive, destination):
    destination = Path(destination).resolve()
    with tarfile.open(archive, "r:gz") as bundle:
        for member in bundle.getmembers():
            if member.issym() or member.islnk():
                raise RuntimeError(f"Refusing archive link: {member.name}")
            target = (destination / member.name).resolve()
            if target != destination and destination not in target.parents:
                raise RuntimeError(f"Unsafe archive member: {member.name}")
        bundle.extractall(destination, filter="data")

if WORK_ROOT.exists():
    resolved = WORK_ROOT.resolve()
    assert resolved.parent == Path("/kaggle/working") and not WORK_ROOT.is_symlink()
    shutil.rmtree(WORK_ROOT)
SOURCE_UNPACK = WORK_ROOT / "source"
SDK_DIST = WORK_ROOT / "dist"
SDK_TARGET = WORK_ROOT / "sdk"
SOURCE_ARCHIVE = WORK_ROOT / "reviewed-source.tar.gz"
SOURCE_UNPACK.mkdir(parents=True)
SDK_DIST.mkdir()
urllib.request.urlretrieve(SOURCE_URL, SOURCE_ARCHIVE)
print("Source archive SHA256:", sha256_file(SOURCE_ARCHIVE))
safe_extract(SOURCE_ARCHIVE, SOURCE_UNPACK)
roots = [path for path in SOURCE_UNPACK.iterdir() if path.is_dir()]
assert len(roots) == 1
SOURCE_ROOT = roots[0]
subprocess.run([sys.executable, "-m", "pip", "install", "build>=1.2"], check=True)
subprocess.run([sys.executable, "-m", "build", "--wheel", "--outdir", str(SDK_DIST), str(SOURCE_ROOT)], check=True)
wheels = sorted(SDK_DIST.glob("kaggle_vllm-*.whl"))
assert len(wheels) == 1
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "--target", str(SDK_TARGET), str(wheels[0])], check=True)
sys.path.insert(0, str(SDK_TARGET))
SDK_ENV = os.environ.copy()
SDK_ENV["PYTHONPATH"] = str(SDK_TARGET)
print("SDK wheel SHA256:", sha256_file(wheels[0]))

In [ ]:
from kaggle_vllm.bootstrap import activate_runtime, bootstrap
from kaggle_vllm.doctor import run_doctor
from kaggle_vllm.environment import as_json

bootstrap_result = bootstrap(strict=True)
activate_runtime(bootstrap_result.manifest)
assert run_doctor(strict=True) == 0
print(as_json())
subprocess.run(["nvidia-smi"], check=True)
subprocess.run(["nvidia-smi", "topo", "-m"], check=True)
SDK_ENV.update(os.environ)
print("Immutable native runtime bootstrap and strict doctor: PASS")

In [ ]:
if ATTACHED_MODEL_PATH is not None:
    MODEL_PATH = Path(ATTACHED_MODEL_PATH).resolve()
    assert Path("/kaggle/input") in MODEL_PATH.parents
    print("Using attached read-only Transformers model:", MODEL_PATH)
else:
    try:
        from huggingface_hub import snapshot_download
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "huggingface_hub>=0.36"], check=True)
        from huggingface_hub import snapshot_download
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
        print("HF_TOKEN available:", bool(token))
    except Exception as error:
        token = None
        print("HF_TOKEN available: False (", type(error).__name__, ")")
    MODEL_PATH = Path(snapshot_download(repo_id=MODEL_REPOSITORY, revision=MODEL_REVISION, token=token, cache_dir=str(WORK_ROOT / "model-cache"))).resolve()
print("Resolved model path:", MODEL_PATH)
assert (MODEL_PATH / "config.json").is_file()
assert not any(MODEL_PATH.glob("model-rank-*-part-*.safetensors")), "TP-specific sharded_state is forbidden"

In [ ]:
RUNNER = SOURCE_ROOT / "scripts" / "kaggle_concurrency_crossover.py"
COMMON = ["--model", str(MODEL_PATH), "--model-source", "local_transformers", "--model-revision", MODEL_REVISION, "--output-dir", str(EVIDENCE_DIR)]
print("Side-effect-free benchmark plan:")
subprocess.run([sys.executable, str(RUNNER), "--dry-run", *COMMON], check=True, env=SDK_ENV)

In [ ]:
subprocess.run([sys.executable, str(RUNNER), "--source-identity", REVIEWED_SOURCE_COMMIT, *COMMON], check=True, env=SDK_ENV)
required = ["run-metadata.json", "environment.json", "topology.txt", "summary.json", "SHA256SUMS.txt"]
assert all((EVIDENCE_DIR / name).is_file() for name in required)
for tp in (1, 2):
    for concurrency in (1, 4, 8, 16, 32, 64):
        stem = f"qwen-tp{tp}-c{concurrency:02d}"
        assert (EVIDENCE_DIR / f"{stem}.json").is_file()
        assert (EVIDENCE_DIR / f"{stem}.server.log").is_file()
        assert (EVIDENCE_DIR / f"{stem}.metrics.txt").is_file()
        assert (EVIDENCE_DIR / f"{stem}.telemetry.jsonl").is_file()
        assert (EVIDENCE_DIR / f"{stem}-requests.jsonl").is_file()
summary = json.loads((EVIDENCE_DIR / "summary.json").read_text())
print(json.dumps(summary, indent=2))

In [ ]:
archive_base = str(EVIDENCE_ZIP.with_suffix(""))
created = Path(shutil.make_archive(archive_base, "zip", root_dir=EVIDENCE_DIR.parent, base_dir=EVIDENCE_DIR.name))
assert created == EVIDENCE_ZIP
print("Evidence ZIP:", EVIDENCE_ZIP)
print("Evidence ZIP SHA256:", sha256_file(EVIDENCE_ZIP))
print("GPU execution completed. Acceptance remains pending independent evidence review.")